---
# Optimisation Clustering Dataset
---

## Library Imports

In [11]:
# ============================================================================
# LIBRARY IMPORTS
# Import all necessary packages for data manipulation, visualization,
# API requests, and statistical analysis.
# ============================================================================

# To handle dataframe objects
import pandas as pd

# To handle time manipulation
from datetime import datetime

# To handle file system operations and temporary files
import tempfile

# To suppress warnings during execution
import warnings

In [12]:
# Suppress all warnings while executing the code to keep output clean
warnings.filterwarnings('ignore')

## Helper Functions

### Utility Functions

In [13]:
def MissingValueSummary(df):
    print('Number of instances = %d' % (df.shape[0]))

    print('\nMissing Values Summary:')
    print('=' * 50)

    # Create a summary DataFrame
    missing_data = pd.DataFrame({
        'Column': df.columns,
        'Missing_Values': [df[col].isnull().sum() for col in df.columns],
        'Percent_Missing': [(df[col].isnull().sum() / len(df)) * 100 
                        for col in df.columns]
    })

    # Print formatted table
    print(missing_data.to_string(index=False, formatters={
        'Percent_Missing': '{:.2f}%'.format
    }))

In [14]:
def TempCSVFile(df):
    temp_csv = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False)
    df.to_csv(temp_csv.name, index=False)
    temp_csv.close()

    print(f"Data exported to: {temp_csv.name}")

## Dataset Imports

In [15]:
borrowings = pd.read_csv("../../../data/processed/library_borrowings.csv")
inventory = pd.read_csv("../../../data/processed/library_inventory.csv")

---
# Data Aggregation
---

In [16]:
# ============================================================================
# AGGREGATE DATAFRAME CREATION
# Create an aggregated dataframe with book-level statistics
# ============================================================================

CURRENT_YEAR = datetime.now().year

# Step 1: Copy borrowings and remove Reader_num
borrowings_copy = borrowings.drop(columns=['Reader_num']).copy()

# Step 2: Prepare inventory data - convert Edition Date to numeric
inventory_clean = inventory.copy()
inventory_clean['Edition Date'] = pd.to_numeric(inventory_clean['Edition Date'], errors='coerce')

# Step 3: Get inventory info grouped by Title (for author, copy count, avg edition date)
inventory_grouped = inventory_clean.groupby('Title').agg({
    'Author': 'first',
    'Edition Date': 'mean',  # Average edition date for age calculation
    'Inventory Number': 'count'  # Copy count
}).reset_index()

inventory_grouped.columns = ['Title', 'Author', 'edition_year', 'copy_count']

# Calculate age as CURRENT_YEAR - edition_year
inventory_grouped['age'] = CURRENT_YEAR - inventory_grouped['edition_year']
inventory_grouped = inventory_grouped.drop(columns=['edition_year'])

# Step 4: Aggregate borrowings by Title
borrowings_agg = borrowings.groupby('Title').agg({
    'category': 'first',
    'borrowing_duration': ['count', 'mean', 'std'],
    'Reader_num': 'nunique'
}).reset_index()

# Flatten multi-level columns
borrowings_agg.columns = ['Title', 'category', 'borrowings_count', 'borrowings_duration_avg', 
                            'borrowings_duration_std', 'unique_readers']

# Step 5: Merge borrowings aggregation with inventory info
aggregate = borrowings_agg.merge(inventory_grouped, on='Title', how='left')

# Step 6: Calculate derived metrics
# Age (average edition date year) - fill missing with mean
aggregate['age'] = aggregate['age'].fillna(aggregate['age'].mean())

# Fill NaN in std deviation with 0 (for books with single borrowing)
aggregate['borrowings_duration_std'] = aggregate['borrowings_duration_std'].fillna(0)

# Fill missing copy_count with 1 (minimum assumption)
aggregate['copy_count'] = aggregate['copy_count'].fillna(1)

# Reorder columns for clarity
aggregate = aggregate[['Title', 
                       'Author', 
                       'category', 
                       'age', 
                       'borrowings_count', 
                       'borrowings_duration_avg', 
                       'borrowings_duration_std',
                       'unique_readers', 
                       'copy_count',
                       ]]

print(f"Aggregate DataFrame Shape: {aggregate.shape}")
print(f"\nColumn Types:\n{aggregate.dtypes}")

TempCSVFile(aggregate)
MissingValueSummary(aggregate)


Aggregate DataFrame Shape: (132, 9)

Column Types:
Title                          str
Author                         str
category                     int64
age                        float64
borrowings_count             int64
borrowings_duration_avg    float64
borrowings_duration_std    float64
unique_readers               int64
copy_count                   int64
dtype: object
Data exported to: C:\Users\mustapha\AppData\Local\Temp\tmp6jn7_kia.csv
Number of instances = 132

Missing Values Summary:
                 Column  Missing_Values Percent_Missing
                  Title               0           0.00%
                 Author               0           0.00%
               category               0           0.00%
                    age               0           0.00%
       borrowings_count               0           0.00%
borrowings_duration_avg               0           0.00%
borrowings_duration_std               0           0.00%
         unique_readers               0           

---
# Feature Engineering
---

### Book Optimization Metrics

#### Objective
We aim to create engineered metrics that better describe book optimization objectives for collection management.

#### Core Metrics

##### 1. Demand Per Copy (DPC)
Measures borrowing intensity relative to available copies.

$$
\text{DPC} = \frac{\text{Borrowing Count}}{\text{Copy Count}}
$$

##### 2. Reader Pressure (RP)
Captures reader demand diversity relative to copy availability.

$$
\text{RP} = \frac{\text{Unique Readers}}{\text{Copy Count}}
$$

##### 3. Utilization Score (U)
A comprehensive metric combining multiple factors, where:
- $\mu$ = average borrowing duration  
- $\sigma$ = standard deviation of borrowing duration  
- $a$ = age penalty

$$
U = \frac{\text{DPC} \times \text{RP} \times \mu}{(1 + \sigma) \times a}
$$

In [17]:
# Demand per copy = borrowings count / copy count
aggregate['demand_per_copy'] = aggregate['borrowings_count'] / aggregate['copy_count']

# Reader pressure = unique readers / copy count
aggregate['reader_pressure'] = aggregate['unique_readers'] / aggregate['copy_count']

In [18]:
dpc = aggregate['demand_per_copy']
rp = aggregate['reader_pressure']
mu = aggregate['borrowings_duration_avg']
sigma = aggregate['borrowings_duration_std']
age = aggregate['age']

aggregate['utilization_score'] = (
    dpc * 
    rp * 
    mu 
) / (
    (sigma + 1) * 
    (age)
)

aggregate = aggregate[['Title', 'Author', 'category', 'age', 
                       'borrowings_count', 
                       'borrowings_duration_avg',
                       'borrowings_duration_std', 
                       'unique_readers', 'copy_count',
                       'demand_per_copy', 'reader_pressure', 'utilization_score']]

In [19]:
TempCSVFile(aggregate)
MissingValueSummary(aggregate)

Data exported to: C:\Users\mustapha\AppData\Local\Temp\tmp8tyht6na.csv
Number of instances = 132

Missing Values Summary:
                 Column  Missing_Values Percent_Missing
                  Title               0           0.00%
                 Author               0           0.00%
               category               0           0.00%
                    age               0           0.00%
       borrowings_count               0           0.00%
borrowings_duration_avg               0           0.00%
borrowings_duration_std               0           0.00%
         unique_readers               0           0.00%
             copy_count               0           0.00%
        demand_per_copy               0           0.00%
        reader_pressure               0           0.00%
      utilization_score               0           0.00%


---
## Exporting the dataframe to a CSV file
---

In [ ]:
# Export to specific path
file_path = r"../../../data/aggregated/clustering-1/aggregate.csv"  # Windows

aggregate.to_csv(file_path, index=False, encoding='utf-8')
print(f"Data exported to: {file_path}")

Data exported to: ../../../data/aggregated/clustering-1/aggregate.csv
